# Redommendation systems

In [6]:
pip install mlxtend

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 6.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

## step 1: Load Dataset

In [8]:
df = pd.read_csv("tourism_dataset.csv")

print(df.head())
print(df.shape)
print(df.info())

     Location Country    Category  Visitors  Rating    Revenue  \
0  kuBZRkVsAR   India      Nature    948853    1.32   84388.38   
1  aHKUXhjzTo     USA  Historical    813627    2.01  802625.60   
2  dlrdYtJFTA  Brazil      Nature    508673    1.42  338777.11   
3  DxmlzdGkHK  Brazil  Historical    623329    1.09  295183.60   
4  WJCCQlepnz  France    Cultural    124867    1.43  547893.24   

  Accommodation_Available  
0                     Yes  
1                      No  
2                     Yes  
3                     Yes  
4                      No  
(5989, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5989 entries, 0 to 5988
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Location                 5989 non-null   object 
 1   Country                  5989 non-null   object 
 2   Category                 5989 non-null   object 
 3   Visitors                 5989 non-null   int

## Step 2: Data Preprocessing

In [9]:
df = df.dropna()

df['Accommodation_Available'] = df['Accommodation_Available'].map({'Yes':1,'No':0})

In [10]:
df['text_features'] = df['Category'] + " " + df['Country']

print(df[['Location','text_features']].head())

     Location      text_features
0  kuBZRkVsAR       Nature India
1  aHKUXhjzTo     Historical USA
2  dlrdYtJFTA      Nature Brazil
3  DxmlzdGkHK  Historical Brazil
4  WJCCQlepnz    Cultural France


## step3: NLP Feature Extraction (TF-IDF)

Convert text into numerical vectors.

In [11]:
tfidf = TfidfVectorizer(stop_words='english')

X_text = tfidf.fit_transform(df['text_features'])

print(X_text.shape)

(5989, 13)


## step 4: Clustering Destinations

Use KMeans clustering to group similar tourist places.

In [12]:
kmeans = KMeans(n_clusters=6, random_state=42)

df['Cluster'] = kmeans.fit_predict(X_text)

print(df[['Location','Category','Cluster']].head())

     Location    Category  Cluster
0  kuBZRkVsAR      Nature        1
1  aHKUXhjzTo  Historical        0
2  dlrdYtJFTA      Nature        4
3  DxmlzdGkHK  Historical        3
4  WJCCQlepnz    Cultural        5


Example clusters:

In [14]:
# Cluster 0 -> Nature tourism
# Cluster 1 -> Cultural tourism
# Cluster 2 -> Historical places
# Cluster 3 -> Adventure tourism
# Cluster 4 -> Religious tourism
# Cluster 5 -> Urban tourism

## step 5:Evaluate Clustering

In [15]:
score = silhouette_score(X_text, df['Cluster'])

print("Silhouette Score:", score)

Silhouette Score: 0.17230916610708985


## step 6:Association Rules (Travel Patterns)

Simulate tourist travel transactions using categories.

In [16]:
#Nature → Cultural
#Historical → Cultural
#Nature → Adventure

In [17]:
transactions = df.groupby('Country')['Category'].apply(list)
transactions = transactions.tolist()

In [18]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_assoc = pd.DataFrame(te_array, columns=te.columns_)

In [19]:
frequent_items = apriori(df_assoc, min_support=0.1, use_colnames=True)

rules = association_rules(frequent_items, metric="confidence", min_threshold=0.5)

print(rules[['antecedents','consequents','support','confidence']])

      antecedents                                       consequents  support  \
0     (Adventure)                                           (Beach)      1.0   
1         (Beach)                                       (Adventure)      1.0   
2     (Adventure)                                        (Cultural)      1.0   
3      (Cultural)                                       (Adventure)      1.0   
4    (Historical)                                       (Adventure)      1.0   
..            ...                                               ...      ...   
597       (Beach)  (Urban, Adventure, Historical, Cultural, Nature)      1.0   
598   (Adventure)      (Urban, Beach, Historical, Cultural, Nature)      1.0   
599  (Historical)       (Urban, Beach, Adventure, Cultural, Nature)      1.0   
600    (Cultural)     (Urban, Beach, Adventure, Historical, Nature)      1.0   
601      (Nature)   (Urban, Beach, Adventure, Historical, Cultural)      1.0   

     confidence  
0           1.0  
1  

c:\Users\Avunoorihinduvahini\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


## step 7:Recommendation System

In [20]:
def recommend_destination(place):

    place_cluster = df[df['Location']==place]['Cluster'].values[0]

    recommendations = df[df['Cluster']==place_cluster]

    return recommendations[['Location','Category','Country','Rating']].head(5)

## step 8:Test Recommendation

In [21]:
recommend_destination(df['Location'].iloc[10])

,Location,Category,Country,Rating
0,kuBZRkVsAR,Nature,India,1.32
5,IKdhVWFKRc,Cultural,Egypt,2.19
8,OcCopAsiyJ,Cultural,Australia,4.10
9,pXDJPYzTeU,Adventure,India,1.98
10,dUCLjskBYA,Urban,Australia,4.20


## step 9:Improved Recommendation (Cluster + Rating)

In [23]:
def recommend_best(place):

    cluster = df[df['Location']==place]['Cluster'].values[0]

    rec = df[df['Cluster']==cluster]

    rec = rec.sort_values(by='Rating',ascending=False)

    return rec[['Location','Category','Country','Rating']].head(5)

## step 10:Example Recommendation

In [24]:
recommend_best("kuBZRkVsAR")

,Location,Category,Country,Rating
5058,JKQtkdMKEH,Urban,Egypt,5.00
3319,llNplbsNzk,Cultural,Australia,5.00
2498,PKvzPvaNal,Cultural,Egypt,4.99
3546,oQqSgwqiem,Cultural,Australia,4.99
5885,cgyNjGIqTr,Nature,India,4.99
